# mine-tuning-model

## 필수 라이브러리 설치

In [1]:
!pip install --upgrade chromadb langchain langchain-community langchain-openai sentence-transformers transformers datasets gradio langgraph -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip freeze > /content/drive/MyDrive/Colab Notebooks/Github/mine-tuning-model/requirements.txt

## 허깅페이스에서 데이터 가져오기

In [2]:
from datasets import load_dataset

ds = load_dataset("lparkourer10/minecraft-wiki")
print(ds)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/371 [00:00<?, ?B/s]

dataset.json:   0%|          | 0.00/49.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87934 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['url', 'question', 'answer'],
        num_rows: 87934
    })
})


In [3]:
# 샘플 3개 출력
for i in range(3):
    print(f"=== 샘플 {i+1} ===")
    print(f"URL   : {ds['train']['url'][i]}")
    print(f"Q     : {ds['train']['question'][i]}")
    print(f"A     : {ds['train']['answer'][i][:200]}")  # 앞 200자만
    print()

=== 샘플 1 ===
URL   : https://minecraft.wiki/w/Minecraft_(franchise)
Q     : What are the main differences between the original Minecraft gameplay and its current version in Windows 10 Edition?
A     : The main difference between the original Minecraft gameplay and its current version in Windows 10 Edition is the addition of new features and game modes, which have been implemented through Microsoft'

=== 샘플 2 ===
URL   : https://minecraft.wiki/w/Minecraft_(franchise)
Q     : Is the Creative mode in Minecraft really as easy to understand and use as it is portrayed in the summary?
A     : No, the creative mode in Minecraft is not entirely easy to understand and use as portrayed in the summary. In fact, some aspects of creative mode can be challenging for new players to grasp. For examp

=== 샘플 3 ===
URL   : https://minecraft.wiki/w/Minecraft_(franchise)
Q     : How does the addition of new biomes, mobs, and game modes in the recent update enhance the overall gameplay experience for player

# 데이터 임베딩


In [4]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print('[입베딩 성공]')
print(f"임베딩 차원: {embedding_model.get_sentence_embedding_dimension()}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[입베딩 성공]
임베딩 차원: 384


/tmp/ipykernel_1813/1680344745.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"임베딩 차원: {embedding_model.get_sentence_embedding_dimension()}")


## Chroma DB 구축

In [5]:
import chromadb

# ChromaDB 초기화
chroma_client = chromadb.Client()

# 이미 있다면 삭제 (없을 때 에러 방지를 위해 try-except 사용)
try:
    chroma_client.delete_collection(name="minecraft_rag")
except:
    pass

collection = chroma_client.create_collection(name="minecraft_rag")

# 데이터 5000개 테스트
sample_size = 5000
answers   = ds["train"]["answer"][:sample_size]
questions = ds["train"]["question"][:sample_size]
urls      = ds["train"]["url"][:sample_size]

# 임베딩 생성 및 저장 (배치 처리)batch_size = 500
batch_size = 500
for i in range(0, sample_size, batch_size):
    batch_answers = answers[i:i+batch_size]
    embeddings = embedding_model.encode(batch_answers, show_progress_bar=True).tolist()

    collection.add(
        documents=batch_answers,
        embeddings=embeddings,
        metadatas=[{"question": q, "url": u} for q, u in zip(questions[i:i+batch_size], urls[i:i+batch_size])],
        ids=[str(j) for j in range(i, i+len(batch_answers))]
    )
    print(f" {min(i+batch_size, sample_size)}/{sample_size} 완료")

print(f"\n{collection.count()}개 저장됨")

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

 500/5000 완료


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

 1000/5000 완료


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

...

 4000/5000 완료


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

 4500/5000 완료


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

 5000/5000 완료

5000개 저장됨


## 검색 테스트

In [6]:
def retrieve(query: str, top_k: int = 3):
    query_embedding = embedding_model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )
    return results["documents"][0], results["metadatas"][0]

# 테스트
docs, metas = retrieve("How to find diamonds?")
for i, (doc, meta) in enumerate(zip(docs, metas)):
    print(f"=== 검색 결과 {i+1} ===")
    print(f"URL     : {meta['url']}")
    print(f"Q       : {meta['question']}")
    print(f"Answer  : {doc[:200]}")
    print()

=== 검색 결과 1 ===
URL     : https://minecraft.wiki/w/Bottle_o%27_Enchanting
Q       : How do players obtain resources such as diamonds and obsidian naturally in the wilderness?
Answer  : In Minecraft, players can naturally obtain diamonds and obsidian by mining them from deep within the game's underground structures, such as caves. Diamonds are typically found in areas with low light 

=== 검색 결과 2 ===
URL     : https://minecraft.wiki/w/Saddle
Q       : What is the process of obtaining certain materials such as diamonds and emeralds that are required for crafting items like bows and arrows?
Answer  : In Minecraft, obtaining certain materials such as diamonds and emeralds typically requires mining them from the ground. Diamonds can be found in deep stone layers, while emeralds are often hidden with

=== 검색 결과 3 ===
URL     : https://minecraft.wiki/w/Leaves
Q       : How do Diamonds affect gameplay or resource gathering in Minecraft?
Answer  : Diamonds are used to craft tools such as pickax

## LLM 로드

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

llm = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False
)

print("LLM 로드 완료")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM 로드 완료


## RAG 파이프라인 완성

In [8]:
def rag_answer(query: str) -> str:
    # 1. 관련 문서 검색
    docs, metas = retrieve(query, top_k=3)
    context = "\n\n".join(docs)

    # 2. 프롬프트 구성
    prompt = f"""You are a helpful Minecraft expert assistant.
Use the following context to answer the question accurately.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {query}
Answer:"""

    # 3. LLM 답변 생성
    messages = [{"role": "user", "content": prompt}]
    result = llm(messages)
    answer = result[0]["generated_text"][-1]["content"]

    # 4. 출처 출력
    print(f"출처: {metas[0]['url']}\n")
    return answer


In [9]:
# 테스트
response = rag_answer("How to find diamonds?")
print(response)

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


출처: https://minecraft.wiki/w/Bottle_o%27_Enchanting

To find diamonds in Minecraft, you need to mine them from deep within the game's underground structures, such as caves. Diamonds are typically found in areas with low light levels, like caves or dark biomes. Additionally, you can explore abandoned mineshafts, which contain both diamonds and obsidian.


# 한국어 번역 파이프 라인

- 번역시에 시간이 너무 오래 걸림.
-> 사용을 하지 않거나, 해결을 해야함



In [12]:
from transformers import MarianMTModel, MarianTokenizer

# 한국어 -> 영어
ko2en_tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-ko-en")
ko2en_model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-ko-en")

# 영어 -> 한국어
en2ko_tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-tc-big-en-ko")
en2ko_model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-tc-big-en-ko")

def translate_ko2en(text: str) -> str:
    inputs = ko2en_tokenizer(text, return_tensors="pt", padding=True)
    outputs = ko2en_model.generate(**inputs)
    return ko2en_tokenizer.decode(outputs[0], skip_special_tokens=True)

def translate_en2ko(text: str) -> str:
    inputs = en2ko_tokenizer(text, return_tensors="pt", padding=True)
    outputs = en2ko_model.generate(**inputs)
    return en2ko_tokenizer.decode(outputs[0], skip_special_tokens=True)


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/790k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/815k [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/418M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

✅ 번역 모델 로드 완료!
How do you find a diamond?


In [15]:
def translate_en2ko_long(text: str) -> str:
    # 문장 단위로 분할해서 번역
    sentences = text.replace("。", ".").split(". ")
    translated = []
    for sent in sentences:
        sent = sent.strip()
        if not sent:
            continue
        try:
            inputs = en2ko_tokenizer(sent, return_tensors="pt", padding=True, truncation=True, max_length=512)
            outputs = en2ko_model.generate(**inputs)
            translated.append(en2ko_tokenizer.decode(outputs[0], skip_special_tokens=True))
        except:
            translated.append(sent)  # 번역 실패시 원문 유지
    return " ".join(translated)

def rag_answer_korean(query_ko: str) -> str:
    # 1. 한국어 -> 영어 번역
    query_en = translate_ko2en(query_ko)
    print(f"🔄 번역된 질문: {query_en}")

    # 2. 영어 질문으로 RAG 검색
    docs, metas = retrieve(query_en, top_k=3)
    context = "\n\n".join(docs)

    # 3. 프롬프트 구성
    prompt = f"""You are a helpful Minecraft expert assistant.
Use the following context to answer the question accurately.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {query_en}
Answer:"""

    # 4. LLM 답변 생성
    messages = [{"role": "user", "content": prompt}]
    result = llm(messages)
    answer_en = result[0]["generated_text"][-1]["content"]
    print(f"📝 영어 답변: {answer_en}\n")

    # 5. 영어 답변 -> 한국어 번역 (문장 단위)
    answer_ko = translate_en2ko_long(answer_en)

    # 6. 출처 출력
    print(f"출처: {metas[0]['url']}\n")
    return answer_ko

# 테스트
response = rag_answer_korean("다이아몬드를 어떻게 찾아?")
print(f"🇰🇷 한국어 답변:\n{response}")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔄 번역된 질문: How do you find a diamond?
📝 영어 답변: To find a diamond in Minecraft, you need to mine it from deep within the game's underground structures. This includes:

1. **Caves**: Look for areas with low light levels where diamonds are commonly found.
2. **Lava Flows**: Obsidian is often formed around lava flows, so look for areas with active lava.
3. **Volcanic Activity**: Explore regions with frequent volcanic eruptions or ash deposits.
4. **Underwater Caves**: Dive into underwater caves to find diamonds that have fallen through the water.
5. **Abandoned Mineshafts**: Dig into old mineshafts to uncover hidden treasures including diamonds and obsidian.

Mining diamonds requires using diamond tools, which are essential for efficient extraction from deeper layers of the earth.

출처: https://minecraft.wiki/w/Bottle_o%27_Enchanting

🇰🇷 한국어 답변:
로크웰 차이나, 일반 코펜하겐 심장, CD Creator, well166 Hoteling, PI 캔과 함께 그、스펀 #、 지혜、 잠금、 148、 신뢰할 수있는。 완전히 지원합니다. 그들은 GOT 주 shockingani Navi 인기와 TrojaningSupport

## Gradio 연결

In [ ]:
import gradio as gr

def chat(message, history):
    answer = rag_answer(message)
    return answer

demo = gr.ChatInterface(
    fn=chat,
    title="⛏️ Minecraft Guide LLM",
    description="Minecraft Wiki 기반 AI 가이드에게 무엇이든 물어보세요!",
    examples=[
        "How to find diamonds?",
        "How do I defeat the Ender Dragon?",
        "How to make a Nether portal?"
    ],
)

demo.launch(share=True)  # share=True 로 외부 접속 링크 생성

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b78f06ca2efb0604f5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
